# Serving Runtimes Comparison: vLLM · TensorRT-LLM · SGLang

> **Status:** Content notebook — benchmarks require NVIDIA GPU hardware.

## Learning Objectives

By the end of this notebook, you will be able to:

- [ ] Describe the design philosophy of vLLM, TensorRT-LLM, and SGLang
- [ ] Identify the key trade-offs across runtimes (TTFT, throughput, ease of deployment)
- [ ] Choose the right runtime for a given use case and hardware budget
- [ ] Run a benchmark comparing throughput and latency across runtimes
- [ ] Understand TGI (Text Generation Inference) as an alternative

---

## Prerequisites

- [02_serving_with_vllm.ipynb](../02_serving_with_vllm/02_serving_with_vllm.ipynb)
- [03_kv_cache_paged_attention.ipynb](../03_kv_cache_paged_attention/03_kv_cache_paged_attention.ipynb)

---

## 1. Runtime Overview

| Feature | vLLM | TensorRT-LLM | SGLang |
|---------|------|--------------|--------|
| Backend | PyTorch + CUDA kernels | TensorRT engine (NVIDIA-only) | PyTorch + RadixAttention |
| Quantization | AWQ, GPTQ, FP8 | AWQ, GPTQ, FP8, INT8 SmoothQuant | AWQ, GPTQ, FP8 |
| Speculative decoding | ✅ | ✅ | ✅ |
| Prefix caching | ✅ (automatic prefix caching) | ✅ | ✅ (RadixAttention) |
| Multi-modal | ✅ | ✅ | ✅ |
| OpenAI-compatible API | ✅ | ✅ | ✅ |
| Hardware | NVIDIA, AMD (ROCm) | NVIDIA only | NVIDIA, AMD |
| Build complexity | Low | High (engine build required) | Low |
| Best for | General purpose production | Maximum NVIDIA throughput | Structured generation, research |

---

## 2. vLLM

vLLM (Virtual Large Language Model) is the most widely deployed open-source
LLM serving framework. Key innovations:
- **PagedAttention** for efficient KV cache management
- **Continuous batching** — new requests join mid-generation
- **Chunked prefill** — large prefill split across multiple steps
- **Automatic prefix caching** — reuse KV blocks for shared prefixes

```bash
# Install
pip install vllm

# Start server
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Meta-Llama-3-8B-Instruct \
  --port 8000 \
  --dtype bfloat16 \
  --enable-prefix-caching
```

```python
# Call vLLM OpenAI-compatible API
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
response = client.chat.completions.create(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    messages=[{"role": "user", "content": "What is PagedAttention?"}],
    max_tokens=200,
)
print(response.choices[0].message.content)
```

---

## 3. TensorRT-LLM

NVIDIA's TensorRT-LLM compiles models into optimized TensorRT engines.
Significantly higher throughput than vLLM on NVIDIA hardware, but:
- Requires a multi-hour engine build step
- NVIDIA hardware only
- More complex deployment pipeline

```python
# Conceptual TensorRT-LLM workflow
# Step 1: Convert model weights to TRT-LLM format
# tensorrt_llm convert_checkpoint --model_dir llama-3-8b --output_dir trtllm_ckpt

# Step 2: Build engine
# trtllm-build --checkpoint_dir trtllm_ckpt --output_dir trtllm_engine #   --max_batch_size 32 --max_input_len 4096 --max_output_len 1024

# Step 3: Serve
# python -m tensorrt_llm.serve --engine_dir trtllm_engine --port 8001
```

---

## 4. SGLang

SGLang (Structured Generation Language) focuses on structured generation,
multi-call programs, and research workloads.

Key innovation: **RadixAttention** — a radix-tree-based prefix cache that
automatically identifies and shares common prefixes across requests, achieving
higher cache hit rates than vLLM's block-based prefix cache.

Best for:
- Structured outputs (JSON schema, regex)
- Agent programs with many LLM calls sharing long system prompts
- Research and evaluation pipelines

```bash
# Install
pip install "sglang[all]"

# Start server
python -m sglang.launch_server \
  --model-path meta-llama/Meta-Llama-3-8B-Instruct \
  --port 30000 \
  --enable-torch-compile
```

---

## 5. Benchmarking

Use the standard LLM performance benchmark (`vllm benchmark_throughput.py`)
or the `genai-perf` tool from NVIDIA:

```bash
# vLLM throughput benchmark
python benchmarks/benchmark_throughput.py \
  --model meta-llama/Meta-Llama-3-8B-Instruct \
  --dataset sharegpt \
  --num-prompts 1000 \
  --backend vllm

# Key metrics to collect:
# - Throughput: output tokens/second
# - TTFT: time to first token (p50, p95, p99)
# - ITL: inter-token latency
# - Request latency: total request duration
```

---

## 6. Decision Guide

```
Start with vLLM for:          Use TRT-LLM when:          Use SGLang when:
- Fast deployment             - NVIDIA hardware only     - Heavy structured output
- Multi-hardware support      - Max throughput needed    - Agent multi-call programs
- AWQ/GPTQ quantization      - Enterprise NVIDIA stack  - Research/eval pipelines
- Active community support    - Engine build is OK       - Prefix cache optimization
```

---

## Exercises

1. Start a vLLM server and benchmark throughput at batch sizes of 1, 4, 16, and 64.
2. Measure the cache hit rate with prefix caching enabled vs disabled on a workload
   with a long shared system prompt.
3. Compare TTFT (p95) between vLLM and SGLang on the same model and request distribution.

---

## References

- [vLLM documentation](https://docs.vllm.ai)
- [TensorRT-LLM documentation](https://nvidia.github.io/TensorRT-LLM/)
- [SGLang documentation](https://docs.sglang.ai)
- [LLM Inference Survey (2024)](https://arxiv.org/abs/2312.11514)

## What Comes Next

- [07_prefix_caching_chunked_prefill.ipynb](../07_prefix_caching_chunked_prefill/07_prefix_caching_chunked_prefill.ipynb) — Prefix caching and chunked prefill tuning
